<a href="https://colab.research.google.com/github/ranjith070327-hash/localrepo/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [19]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [20]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [21]:
DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section. "
        "Documents can be related to any topic - sports, manga, food, music etc. "
        "Have a minimum of 4 sentences with each document. "
        "Duplicate this for a minimum of 4 documents to test your agent."
    ),

    "cricket": (
        "Cricket is a popular sport played in many countries. "
        "Two teams compete against each other in a cricket match. "
        "Each team tries to score more runs than the opponent. "
        "India has a large number of cricket fans."
    ),

    "technology": (
        "Technology plays an important role in our daily lives. "
        "Computers help people complete tasks quickly. "
        "Artificial intelligence is used in education, healthcare, and business. "
        "Technology continues to develop every year."
    ),

    "food": (
        "Food provides energy and nutrients to our bodies. "
        "South Indian food includes dosa, idli, and sambar. "
        "Rice is a common food in many Indian households. "
        "Eating a balanced diet helps us stay healthy."
    ),

    "music": (
        "Music is enjoyed by people around the world. "
        "There are different types of music, including pop, classical, and rock. "
        "Listening to music can help people relax. "
        "Many people enjoy singing and playing musical instruments."
    ),

    "manga": (
        "Manga refers to Japanese comics and graphic novels. "
        "Manga covers genres such as action, romance, comedy, and adventure. "
        "Many manga stories are adapted into anime series. "
        "People of different ages enjoy reading manga."
    )
}

In [23]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))


[Source: sample_protocol]
Students must fill in documents in this section. Documents can be related to any topic - sports, manga, food, music etc. Have a minimum of 4 sentences with each document. Duplicate this for a minimum of 4 documents to test your agent.

[Source: music]
Music is enjoyed by people around the world. There are different types of music, including pop, classical, and rock. Listening to music can help people relax. Many people enjoy singing and playing musical instruments.


In [25]:
# @title Ask Gemini to search for something

import google.generativeai as genai
import time
import os

# Configure API key
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# Create Gemini model
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",

    tools=[search_documents],

    system_instruction=(
        '''
        You are an intelligent assistant that answers questions
        using the provided documents.

        When you need information about cricket, technology,
        food, music, or manga, use the search_documents tool.

        Only answer based on what you find in the documents.
        Do not use outside knowledge or make up information.

        If the documents do not contain the answer,
        clearly say that the information is not available
        in the provided documents.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-2.5-flash


In [ ]:
import time
from google import genai
from google.genai import types

# Initialize Gemini client
client = genai.Client()

# Use a valid Gemini model
MODEL_NAME = "gemini-3.8-flash"

# Define search tool
search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search the documents to find the most relevant paragraph.",
            parameters={
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"],
            },
        )
    ]
)

# Configure tool
config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
    system_instruction="""
    You are an intelligent assistant that answers questions
    using the provided documents.

    When you need information about cricket, technology,
    food, music, or manga, use the search_documents tool.

    Only answer based on the information found in the documents.
    Do not use outside knowledge.

    If the documents do not contain the answer, clearly say so.
    """
)


# Retry function
def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config
            )

        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


# Run agent
def run_agent(question: str, max_steps: int = 8) -> str:

    print(f"Question: {question}\n" + "-" * 50)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=question)]
        )
    ]

    for step in range(1, max_steps + 1):

        response = send_with_retry(contents)

        model_content = response.candidates[0].content

        # Preserve model response
        contents.append(model_content)

        # Check for function calls
        calls = [
            p.function_call
            for p in model_content.parts
            if p.function_call
        ]

        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Execute tool calls
        result_parts = []

        for call in calls:

            args = dict(call.args)

            print(f"Step {step}: model calls {call.name}({args})")

            # Correct function name
            if call.name == "search_documents":
                result = search_documents(**args)
            else:
                result = "Unknown function."

            print(f"Result preview: {str(result)[:100]}...")

            result_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )

        contents.append(
            types.Content(
                role="user",
                parts=result_parts
            )
        )

    return "(Stopped: hit max_steps)"


# Test the agent
_ = run_agent(
    "What topics are students supposed to share documents on?"
)

Question: What topics are students supposed to share documents on?
--------------------------------------------------


In [ ]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "Add a question here.",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)


In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)